**STEP 1: DATA LOADING**

In [99]:
import pandas as pd

In [100]:
df = pd.read_csv("ambucast_data.csv")

In [101]:
df.columns

Index(['date', 'hour', 'zone_name', 'call_count', 'PM2.5', 'PM10', 'AQI',
       'temperature', 'humidity', 'population_density', 'elderly_pct',
       'day_of_week', 'is_peak_hour'],
      dtype='object')

In [102]:
# # Convert to classification target
# threshold = df["call_count"].quantile(0.60)

# df["high_demand"] = (df["call_count"] >= threshold).astype(int)

**⚙️ STEP 2: FEATURE ENGINEERING**

In [103]:
import numpy as np

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

**📈 Rolling feature**

In [104]:
df = df.sort_values(["zone_name", "date", "hour"])

df["rolling_calls_7"] = (
    df.groupby("zone_name")["call_count"]
    .transform(lambda x: x.rolling(24 * 7, min_periods=1).mean())
)

**📊 Lag feature**

In [105]:
df["lag_24h"] = (
    df.groupby("zone_name")["call_count"]
    .shift(24)
)

In [106]:
df = df.dropna()

**⚙️ STEP 3: FEATURES SELECTION**

In [107]:
features = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "PM2.5", "PM10", "AQI",
    "temperature", "humidity",
    "population_density", "elderly_pct",
    "rolling_calls_7", "lag_24h"
]

x = df[features]
y = df["call_count"]

**STEP 4: TRAIN / TEST SPLIT (TIME-BASED)**

In [108]:
split_date = "2016-07-01"

train = df[df["date"] < split_date]
test  = df[df["date"] >= split_date]

x_train = train[features]
y_train = train["call_count"]

x_test = test[features]
y_test = test["call_count"]

**⚙️ STEP 5: TRAIN XGBOOST**

In [109]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [ ]:
model.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)


**STEP 6: EVALUATION METRICS**

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

**SAMPLE PREDICTIONS**

In [ ]:
def predict_hotspots(input_df):

    # Predict call counts
    preds = model.predict(input_df[features])

    # Fix negatives
    preds = np.clip(preds, 0, None)

    # Convert to risk score (0–100)
    risk_score = (preds / preds.max()) * 100

    # Attach results
    input_df = input_df.copy()
    input_df["predicted_calls"] = preds.round().astype(int)

    input_df["risk_score"] = risk_score.round(2)

    return input_df[["zone_name", "predicted_calls", "risk_score"]]

In [ ]:
result = predict_hotspots(df.sample(20))
result.head()

In [ ]:
result = predict_hotspots(df.sample(25))

# convert zone_name → zone_id
zone_map = {
    "North": "Z01",
    "South": "Z02",
    "East": "Z03",
    "West": "Z04",
    "North East": "Z05",
    "North West": "Z06",
    "South West": "Z07",
    "New Delhi": "Z08"
}

result["zone_id"] = result["zone_name"].map(zone_map)

# keep only needed columns
final = result[["zone_id", "predicted_calls", "risk_score"]]

final.to_json("predictions.json", orient="records")

In [ ]:
final.head(5)

**STEP 7: EXPORT**

In [ ]:
import pickle

with open("hotspotcast.pkl", "wb") as f:
     pickle.dump(model, f)

In [ ]:
import joblib
joblib.dump(model, "hotspotcast.pkl")